# Session 6 — Your turn: From base model to assistant

Two pieces of the preference-training machinery, written with numpy: the reward model's loss on pairs, and what picking the best of N samples buys.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np

rng = np.random.default_rng(4)
r_chosen = rng.normal(0.3, 0.5, 1000)                  # a reward model's scores for the preferred answers
r_rejected = rng.normal(0.0, 0.5, 1000)                # and for the rejected ones
gold = rng.normal(0, 1, (50, 64))                      # a trusted score for 64 sampled answers to each of 50 prompts
proxy = 0.7 * gold + np.sqrt(1 - 0.7 ** 2) * rng.normal(0, 1, gold.shape)       # a cheaper reward model that agrees with it only partly
from aij_check import check, close, summary

## Task 1: The pairwise reward loss

A reward model is trained so that the preferred answer scores higher: the loss for one pair is `-log(sigmoid(r_chosen - r_rejected))`. Compute `pair_loss` (an array, one value per pair), `mean_loss` and `pair_accuracy` (the share of pairs where the chosen answer scores higher).
Also store `loss_if_tied`, the loss when both scores are equal.

In [ ]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
pair_loss = ...
mean_loss = ...
pair_accuracy = ...
loss_if_tied = ...

In [ ]:
check("one loss per pair", lambda: np.shape(pair_loss) == (1000,))
check("a tie costs ln 2", lambda: close(loss_if_tied, np.log(2), 1e-12))
check("the pair loss matches the formula", lambda: close(pair_loss, np.log1p(np.exp(-(r_chosen - r_rejected))), 1e-9))
check("the accuracy is the share of correctly ordered pairs", lambda: close(pair_accuracy, (r_chosen > r_rejected).mean(), 1e-12) and 0.6 < pair_accuracy < 0.75)
check("the mean loss is below ln 2 because the scores carry some signal", lambda: mean_loss < np.log(2))
summary()

## Task 2: Best of N against a proxy

For N in 1, 4, 16, 64: for each prompt, draw N of its 64 answers at random (use `np.random.default_rng(0)`, 200 draws per prompt), pick the one the **proxy** scores highest and record its **gold** score; average over prompts and draws.
Store the averages in `by_proxy[N]`. Also store `oracle[N]`, the average gold score of the best answer by gold among the same N draws (what a perfect judge would get).

In [ ]:
gen = np.random.default_rng(0)
by_proxy, oracle = {}, {}
for n in (1, 4, 16, 64):
    ...

In [ ]:
check("N = 1 is a single random answer, with a gold score near the average (0)", lambda: abs(by_proxy[1]) < 0.1)
check("the proxy's picks improve with N", lambda: by_proxy[1] < by_proxy[4] < by_proxy[16] < by_proxy[64])
check("a perfect judge always does at least as well", lambda: all(oracle[n] >= by_proxy[n] - 1e-9 for n in (1, 4, 16, 64)))
check("the gap to the perfect judge widens with N", lambda: oracle[64] - by_proxy[64] > oracle[4] - by_proxy[4])
summary()